# `Cache` - Mise en cache des données agricoles

**Module :** `kadi.kidas.cache`  
**Classe :** `Cache`

---

## Introduction

`Cache` est le moteur de persistance du module kidas. Il évite de recharger et retraiter des données déjà acquises :

- **Stockage SQLite** : dans `~/.kadi/kidas_cache/` (configurable)
- **Compression zlib** : économie de 60–80% d'espace disque sur les DataFrames
- **Versioning SHA256** : détection automatique des modifications de la source
- **Historique** : chaque remplacement archive la version précédente
- **Expiration** : suppression automatique des entrées trop anciennes
- **Inspection** : taille, nombre d'entrées, clés stockées

## 1. Configuration initiale

In [ ]:
import os
import pandas as pd

import os
import tempfile
import pandas as pd
import numpy as np

from kadi.kidas.cache import Cache

# Répertoire temporaire pour les exemples
cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_demo_cache')

# DataFrame agricole de démonstration
df_demo = pd.DataFrame({
    'culture': ['maïs', 'niébé', 'igname', 'sorgho', 'manioc'],
    'marche':  ['Dantokpa', 'Parakou', 'Bohicon', 'Kandi', 'Cotonou'],
    'prix_xof': [350, 500, 250, 300, 180],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1200.0, 3500.0],
    'date': pd.to_datetime(['2024-01-15'] * 5),
})

print(f"Répertoire cache : {cache_dir}")

## 2. Initialisation de `Cache`

```python
Cache(
    cache_dir: str = '~/.kadi/kidas_cache/',
    max_age_days: int = 30,
)
```

In [ ]:
# Initialisation avec répertoire par défaut (~/.kadi/kidas_cache/)
cache_defaut = Cache(cache_dir=cache_dir)
print(f"Répertoire du cache : {cache_defaut.dir}")
print(f"Durée max        : {cache_defaut.ttl} jours")

In [ ]:
# Initialisation avec répertoire et durée personnalisés
cache = Cache(cache_dir=cache_dir, max_age_days=365)
print(repr(cache))

## 3. `set()` - Sauvegarde d'un DataFrame dans le cache

In [ ]:
cache = Cache(cache_dir=cache_dir)

# Sauvegarde avec une clé descriptive
cle = 'prix_marches_benin_jan2024'
succes = cache.set(cle, df_demo)
print(f"Sauvegarde réussie : {succes}")

In [ ]:
# Sauvegarde avec métadonnées optionnelles
metadata_supplementaires = {
    'source': 'MAEP Bénin',
    'region': 'Sud Bénin',
    'nb_marches': 5,
}

succes = cache.set(
    'prix_avec_meta',
    df_demo,
    meta=metadata_supplementaires,
)
print(f"Sauvegarde avec métadonnées : {succes}")

## 4. `get()` - Chargement d'un DataFrame depuis le cache

In [ ]:
# Chargement par clé
df_charge, meta_charge = cache.get('prix_marches_benin_jan2024')

if df_charge is not None:
    print(f"DataFrame chargé : {len(df_charge)} lignes × {len(df_charge.columns)} colonnes")
    print("\nMétadonnées du cache :")
    for cle_meta, val_meta in meta_charge.items():
        print(f"  {cle_meta:12s} : {val_meta}")
    df_charge

In [ ]:
# Chargement d'une clé inexistante → None
df_absent, meta_absent = cache.get('cle_inexistante')
print(f"Clé absente → df={df_absent}, meta={meta_absent}")

In [ ]:
# Chargement avec métadonnées enrichies
df_meta, meta_meta = cache.get('prix_avec_meta')
print("Métadonnées personnalisées récupérées :")
if meta_meta:
    for k, v in meta_meta.items():
        print(f"  {k} : {v}")

## 5. `keys()` - Lister les clés stockées

In [ ]:
# Ajout de quelques entrées supplémentaires
cache.set('chirps_precip_2024', df_demo.copy())
cache.set('soilgrids_ph_benin', df_demo.copy())

# Listage de toutes les clés
cles = cache.keys()
print(f"Entrées en cache ({len(cles)}) :")
for cle in cles:
    print(f"  - {cle}")

## 6. `size()` - Taille et statistiques du cache

In [ ]:
taille = cache.size()
print("=== Statistiques du cache ===")
for cle, val in taille.items():
    print(f"  {cle:15s} : {val}")

## 7. `history()` - Historique des versions d'une entrée

In [ ]:
# Simulation d'une mise à jour (2ème sauvegarde = archive la 1ère)
df_v2 = df_demo.copy()
df_v2['prix_xof'] = df_v2['prix_xof'] * 1.05  # +5% inflation

cache.set('prix_marches_benin_jan2024', df_v2)  # remplace la v1

# Consultation de l'historique
historique = cache.history('prix_marches_benin_jan2024')
print(f"Versions archivées : {len(historique)}")
for i, version in enumerate(historique):
    print(f"  Version {i+1} : {version.get('archived_at', 'N/A')}")

## 8. `delete()` - Suppression d'une entrée du cache

In [ ]:
# Suppression d'une entrée spécifique
succes_inv = cache.delete('chirps_precip_2024')
print(f"Entrée 'chirps_precip_2024' supprimée : {succes_inv}")

# Vérification
cles_restantes = cache.keys()
print(f"Clés restantes : {cles_restantes}")

## 9. `purge()` - Purge des entrées expirées

In [ ]:
# Suppression des entrées de plus de 365 jours
nb_supprimes = cache.purge(days=365)
print(f"Entrées supprimées (>365 jours) : {nb_supprimes}")

# Avec 0 jours → supprime tout (test de nettoyage)
nb_tout = cache.purge(days=0)
print(f"Entrées supprimées (>0 jours)   : {nb_tout}")

## 10. `clear()` - Vider complètement le cache

In [ ]:
# Rechargement de quelques entrées
cache.set('entree_1', df_demo)
cache.set('entree_2', df_demo)
print(f"Avant clear : {len(cache.keys())} entrées")

# Vidage complet
cache.clear()
print(f"Après clear : {len(cache.keys())} entrées")

## 11. Pattern Cache-aside (recommandé)

Stratégie recommandée : charger depuis le cache si disponible, sinon acquérir et sauvegarder.

In [ ]:
def charger_donnees_marche(marche: str, annee: int, cache: Cache) -> pd.DataFrame:
    """Charge les données d'un marché depuis le cache ou les génère.

    Args:
        marche (str): Nom du marché cible.
        annee (int): Année des données.
        cache (Cache): Instance du cache kidas.

    Returns:
        pd.DataFrame: Données du marché (depuis le cache ou fraîchement acquises).
    """
    # Construction de la clé de cache unique
    cle = f"marche_{marche.lower()}_{annee}"

    # Tentative de chargement depuis le cache
    df_cached, _ = cache.get(cle)
    if df_cached is not None:
        print(f"Cache HIT pour '{cle}'")
        return df_cached

    # Cache MISS : acquisition des données (simulation)
    print(f"Cache MISS pour '{cle}' → acquisition en cours...")
    df_nouveau = pd.DataFrame({
        'culture': ['maïs', 'niébé', 'igname'],
        'marche': [marche] * 3,
        'annee': [annee] * 3,
        'prix_xof': [350, 500, 250],
    })

    # Sauvegarde dans le cache
    cache.set(cle, df_nouveau)
    return df_nouveau


# Démonstration du pattern
cache = Cache(cache_dir=cache_dir)

# 1er appel : Cache MISS (acquisition)
df_dantokpa = charger_donnees_marche('Dantokpa', 2024, cache)
print()

# 2ème appel : Cache HIT (lecture directe)
df_dantokpa_v2 = charger_donnees_marche('Dantokpa', 2024, cache)
print(f"\nDataFrame chargé : {len(df_dantokpa_v2)} lignes")

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `set(key, df, metadata)` | Sauvegarder un DataFrame (pickle + zlib) |
| `get(key)` | Charger un DataFrame et ses métadonnées |
| `keys()` | Lister toutes les clés stockées |
| `size()` | Statistiques d'occupation du cache |
| `history(key)` | Historique des versions pour une clé |
| `delete(key)` | Supprimer une entrée spécifique |
| `purge(days)` | Purger les entrées expirées |
| `clear()` | Vider complètement le cache |